In [ ]:
# Data preparation — cohort construction for dementia detection and future-incidence prediction
#
# Input : master CSV with one row per fundus image, and the AutoMorph M0 image folder
#         Columns used: pngfilename, PAT_ID (individual), STDY_KEY (health-checkup visit), STDY_DT (examination date),
#         SEX, STDY_AGE, sbp, bmi, cholesterol_updated, EXERCISE_STATUS, eye_orientation,
#         days_diff (dementia diagnosis date − examination date, in days; empty if never diagnosed)
# Output: dementia_detection.csv  — development dataset (prevalent dementia vs control, train/valid/test flags)
#         dementia_prediction.csv — future-incidence cohort (event, obs_time)

import os
import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.model_selection import StratifiedShuffleSplit


# -------------------------------------------------------------------------
# Paths — update these to match your environment
# -------------------------------------------------------------------------
MASTER_CSV = "/path/to/master.csv"
IMG_DIR    = "/path/to/AutoMorph/M0/images/"     # fundus PNGs (AutoMorph M0 output)
DET_CSV    = "/path/to/dementia_detection.csv"
PRED_CSV   = "/path/to/dementia_prediction.csv"

RS = 373777                                      # seed for control sampling and the data split
END_OF_FOLLOW_UP = pd.Timestamp("2019-01-01")    # follow-up until 31 December 2018
MAX_FOLLOW_UP    = 3650                          # days (10 years after the index examination)


# -------------------------------------------------------------------------
# Utilities
# -------------------------------------------------------------------------
def sanitize(fn):
    """Replace dots in the base file name with underscores (to match the file names in IMG_DIR)."""
    b, e = os.path.splitext(fn.strip())
    return b.replace(".", "_") + e


def calc_caide_napoe(frame):
    """CAIDE dementia risk score without APOE; education is unavailable and scored 0 (attainable range 0–12).
    Required columns: STDY_AGE, SEXINT, sbp, bmi, cholesterol_updated, EXERCISE_STATUS."""
    x = frame.copy()
    for col in ["STDY_AGE", "SEXINT", "sbp", "bmi", "cholesterol_updated", "EXERCISE_STATUS"]:
        x[col] = pd.to_numeric(x[col], errors="coerce") if col in x.columns else np.nan

    age = x["STDY_AGE"]
    age_pts  = np.select([age < 47, (age >= 47) & (age <= 53), age > 53], [0, 3, 4], default=np.nan)
    sex_pts  = np.where(x["SEXINT"] == 1, 1, 0)                              # male: 1
    edu_pts  = np.zeros(len(x), dtype=float)                                 # education not available: 0
    sbp_pts  = np.where(x["sbp"] >= 140, 2, 0)                               # systolic BP >= 140 mmHg: 2
    bmi_pts  = np.where(x["bmi"] >= 30, 2, 0)                                # BMI >= 30 kg/m2: 2
    chol_pts = np.where(x["cholesterol_updated"] * 0.02586 >= 6.5, 2, 0)     # total cholesterol >= 6.5 mmol/L: 2
    pa_pts   = np.where(x["EXERCISE_STATUS"] >= 2, 0, 1)                     # regular activity (>= 2): 0, otherwise 1

    score = age_pts + sex_pts + edu_pts + sbp_pts + bmi_pts + chol_pts + pa_pts
    return pd.Series(score, index=frame.index, name="CAIDE_noAPOE")


def get_caide_valid_mask(frame):
    """True if all six CAIDE components are available."""
    req = ["STDY_AGE", "SEXINT", "sbp", "bmi", "cholesterol_updated", "EXERCISE_STATUS"]
    return frame[req].notna().all(axis=1)


def summarize(name, d):
    print(f"{name:<12} images {len(d):>7,} | visits {d['STDY_KEY'].nunique():>6,} | individuals {d['PAT_ID'].nunique():>6,}")

In [ ]:
df = pd.read_csv(MASTER_CSV, low_memory=False)

# image paths; keep only images present in IMG_DIR
df["pngfilename"] = IMG_DIR + df["pngfilename"].apply(sanitize)
valid_png = {f for f in os.listdir(IMG_DIR) if f.lower().endswith(".png")}
df = df[df["pngfilename"].apply(lambda p: os.path.basename(p) in valid_png)].reset_index(drop=True)

df["STDY_DT"] = pd.to_datetime(df["STDY_DT"])
df["SEXINT"]  = (df["SEX"] == "M").astype(int)
summarize("All", df)

## Detection dataset (prevalent dementia)

In [ ]:
# dementia-positive: diagnosis date on or before the index examination (days_diff <= 0)
df["label"] = (df["days_diff"] <= 0).astype(int)

# controls: images of individuals with no dementia diagnosis, randomly under-sampled to 1:4 (no covariate matching)
pos = df[df.label == 1]
neg = df[df.days_diff.isna() & ~df.PAT_ID.isin(pos.PAT_ID)].sample(n=len(pos) * 4, random_state=RS)
df_bin = pd.concat([pos, neg]).reset_index(drop=True)

df_bin["CAIDE_noAPOE"]       = calc_caide_napoe(df_bin)
df_bin["CAIDE_noAPOE_valid"] = get_caide_valid_mask(df_bin)

summarize("Dementia", pos)
summarize("Control", neg)
summarize("Development", df_bin)

In [ ]:
# 6:2:2 split without overlap of individuals, stratified by whether an individual has a dementia-positive image
g = df_bin.groupby("PAT_ID", as_index=False).label.max()
tr_i, tmp_i = next(StratifiedShuffleSplit(1, test_size=0.4, random_state=RS).split(g.PAT_ID, g.label))
tr_ids = g.iloc[tr_i].PAT_ID
tmp    = g.iloc[tmp_i].reset_index(drop=True)
va_i, te_i = next(StratifiedShuffleSplit(1, test_size=0.5, random_state=RS).split(tmp.PAT_ID, tmp.label))

df_bin["train"] = df_bin.PAT_ID.isin(tr_ids)
df_bin["valid"] = df_bin.PAT_ID.isin(tmp.iloc[va_i].PAT_ID)
df_bin["test"]  = df_bin.PAT_ID.isin(tmp.iloc[te_i].PAT_ID)

train_df = df_bin[df_bin.train]
valid_df = df_bin[df_bin.valid]
test_df  = df_bin[df_bin.test]

pct = lambda d: 100 * d.label.mean()
print(f"TRAIN {len(train_df)} (pos {pct(train_df):.2f}%) | "
      f"VAL {len(valid_df)} (pos {pct(valid_df):.2f}%) | "
      f"TEST {len(test_df)} (pos {pct(test_df):.2f}%)")
for name, d in (("Train", train_df), ("Validation", valid_df), ("Test", test_df)):
    summarize(name, d)

df_bin.to_csv(DET_CSV, index=False)

## Future-incidence cohort

In [ ]:
# individuals not used in the development dataset; none of them has a dementia diagnosis before an examination
used = set(df_bin.PAT_ID)
fut = df[~df.PAT_ID.isin(used)].copy()

# incident dementia: any diagnosis after the index examination. Follow-up runs from the index examination until
# diagnosis, 31 December 2018 or 10 years after the examination, whichever comes first; images with obs_time beyond
# 10 years are counted as censored (event = 0).
fut["event"] = (fut["days_diff"] > 0).astype(int)
fut["obs_time"] = np.where(fut.event == 1,
                           fut["days_diff"],
                           (END_OF_FOLLOW_UP - fut["STDY_DT"]).dt.days)
fut["obs_time"] = fut["obs_time"].astype("int32")
fut.loc[fut.obs_time > MAX_FOLLOW_UP, "event"] = 0

fut["CAIDE_noAPOE"]       = calc_caide_napoe(fut)
fut["CAIDE_noAPOE_valid"] = get_caide_valid_mask(fut)
summarize("Future", fut)

fut.to_csv(PRED_CSV, index=False)

In [ ]:
# follow-up (truncated at 10 years), incident dementia and incidence rate in the future-incidence cohort
t = np.minimum(fut["obs_time"].to_numpy(float), MAX_FOLLOW_UP)
e = fut["event"].to_numpy(int)

q1, med, q3 = np.percentile(t / 365.25, [25, 50, 75])
n_ev, py = int(e.sum()), t.sum() / 365.25
lo, hi = chi2.ppf(0.025, 2 * n_ev) / 2, chi2.ppf(0.975, 2 * n_ev + 2) / 2     # exact Poisson 95% CI of the count

print(f"median follow-up {med:.2f} years (IQR {q1:.2f}–{q3:.2f})")
print(f"incident dementia {n_ev:,} images ({100 * n_ev / len(fut):.1f}%) | {py:,.0f} person-years | "
      f"{1000 * n_ev / py:.1f} per 1,000 person-years (95% CI {1000 * lo / py:.1f}–{1000 * hi / py:.1f})")